In [ ]:
# ==============================================================================
# MÓDULO DE OPERAÇÕES
# ==============================================================================

# Widgets da Tarefa
id_tarefa = pn.widgets.IntInput(name="ID da Tarefa (Alterar/Excluir)", value=0, start=0)
tipo_tarefa = pn.widgets.TextInput(name="📋 Tipo/Descrição da Atividade", value='', placeholder='Ex: Adubação Orgânica, Rega Geral, Poda')
status_tarefa = pn.widgets.Select(name="⚙️ Status Atual", options=['Pendente', 'Em Andamento', 'Concluído'], value='Pendente')
id_usuario_responsavel = pn.widgets.IntInput(name="👤 ID do Usuário Responsável", value=1, start=1)
id_canteiro_alvo = pn.widgets.IntInput(name="🥬 ID do Canteiro Alvo", value=1, start=1)

# Widgets de Consumo de Insumos (Tabela Associativa 'uso')
id_insumo_gasto = pn.widgets.IntInput(name="📦 ID do Insumo Utilizado (Opcional, 0 se nenhum)", value=0, start=0)
qtd_insumo_gasta = pn.widgets.FloatInput(name="🔢 Quantidade Consumida", value=0.0, start=0.0)

# Botões de Ação
btn_consultar_op = pn.widgets.Button(name='Consultar Cronograma', button_type='default')
btn_inserir_op = pn.widgets.Button(name='Agendar Atividade + Insumo', button_type='default')
btn_atualizar_op = pn.widgets.Button(name='Atualizar Status', button_type='default')
btn_excluir_op = pn.widgets.Button(name='Cancelar Atividade', button_type='default')

def queryAllOperacoes():
    """Realiza a junção (JOIN) entre Tarefas, Usuários e o consumo na tabela associativa 'uso'."""
    query = """
        SELECT 
            t.id_tarefa,
            t.tipo AS atividade,
            t.status,
            u.nome_completo AS responsavel,
            cant.id_canteiro AS canteiro,
            i.nome AS insumo_usado,
            uso.quantidade AS qtd_gasta
        FROM tarefa t
        JOIN usuario u ON t.id_usuario = u.id_usuario
        JOIN canteiro cant ON t.id_canteiro = cant.id_canteiro
        LEFT JOIN uso ON t.id_tarefa = uso.id_tarefa
        LEFT JOIN insumos i ON uso.id_insumos = i.id_insumos
        ORDER BY t.id_tarefa DESC;
    """
    df = pd.read_sql_query(query, cnx)
    
    # Trata valores nulos para tarefas que não consumiram insumos
    if not df.empty:
        df['insumo_usado'] = df['insumo_usado'].fillna('Nenhum')
        df['qtd_gasta'] = df['qtd_gasta'].fillna(0.0)
        
    return pn.widgets.Tabulator(df, layout='fit_columns', pagination='remote', page_size=10)

def on_inserir_operacao():
    """Insere a tarefa e, caso um insumo tenha sido associado, alimenta a entidade 'uso'."""
    try:
        cursor = con.cursor()
        
        # 1. Cria a tarefa e recupera o ID serial gerado pelo banco
        query_tarefa = """
            INSERT INTO tarefa (tipo, status, id_usuario, id_canteiro) 
            VALUES (%s, %s, %s, %s) RETURNING id_tarefa;
        """
        cursor.execute(query_tarefa, (tipo_tarefa.value_input, status_tarefa.value, id_usuario_responsavel.value, id_canteiro_alvo.value))
        nova_id_tarefa = cursor.fetchone()[0]
        
        # 2. Se o ID do insumo for válido (> 0) e houver quantidade, registra na tabela associativa 'uso'
        if id_insumo_gasto.value > 0 and qtd_insumo_gasta.value > 0:
            query_uso = "INSERT INTO uso (id_tarefa, id_insumos, quantidade) VALUES (%s, %s, %s);"
            cursor.execute(query_uso, (nova_id_tarefa, id_insumo_gasto.value, qtd_insumo_gasta.value))
            
        con.commit()
        cursor.close()
        pn.state.notifications.success("Atividade operacional registrada!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao agendar atividade: {str(e)}', alert_type='danger')

def on_atualizar_operacao():
    """Atualiza o status ou a descrição de uma atividade em andamento."""
    try:
        cursor = con.cursor()
        query = "UPDATE tarefa SET tipo = %s, status = %s WHERE id_tarefa = %s;"
        cursor.execute(query, (tipo_tarefa.value_input, status_tarefa.value, id_tarefa.value))
        con.commit()
        cursor.close()
pn.state.notifications.success("Status da atividade atualizado!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao atualizar: {str(e)}', alert_type='danger')

def on_excluir_operacao():
    """Remove com segurança os vínculos da tabela de consumo antes de excluir a tarefa mãe."""
    try:
        cursor = con.cursor()
        # 1. Limpa os insumos consumidos por essa tarefa (tabela associativa 'uso')
        cursor.execute("DELETE FROM uso WHERE id_tarefa = %s;", (id_tarefa.value,))
        # 2. Exclui a tarefa fisicamente
        cursor.execute("DELETE FROM tarefa WHERE id_tarefa = %s;", (id_tarefa.value,))
        con.commit()
        cursor.close()
        pn.state.notifications.success("Atividade cancelada e removida do cronograma!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao excluir atividade: {str(e)}', alert_type='danger')

def operacao_table_creator(cons, ins, atu, exc):
    if ins: return on_inserir_operacao()
    if atu: return on_atualizar_operacao()
    if exc: return on_excluir_operacao()
    return queryAllOperacoes()

interactive_table_op = pn.bind(
    operacao_table_creator, 
    btn_consultar_op, btn_inserir_op, btn_atualizar_op, btn_excluir_op
)

# Montagem estrutural do Layout da Tela
modulo_operacoes = pn.Row(
    pn.Column(
        '### 🛠️ Gerenciamento de Atividades',
        id_tarefa,
        tipo_tarefa,
        status_tarefa,
        pn.Row(id_usuario_responsavel, id_canteiro_alvo),
        '### 🧺 Consumo de Materiais (M-N)',
        pn.Row(id_insumo_gasto, qtd_insumo_gasta),
        pn.Row(btn_consultar_op, btn_inserir_op),
        pn.Row(btn_atualizar_op, btn_excluir_op),
        width=380
    ),
    pn.Column(
        "## 📅 Cronograma de Manutenção e Auditoria de Insumos",
        interactive_table_op,
        margin=(10, 0, 0, 20)
    )
)

modulo_operacoes